In [1]:
# GPU 확인
!nvidia-smi -L || echo "GPU 미할당 - 런타임 유형을 GPU로 바꿔주세요"


GPU 0: NVIDIA L4 (UUID: GPU-d2513970-891f-a7ee-72c8-94286ebcf989)


In [2]:
# 의존성 설치
# mediapipe는 버전을 pin하지 않습니다 (최신 Tasks API만 사용, mp.solutions 미사용).
# 예전에 mediapipe==0.10.13처럼 구버전을 고정하면 protobuf가 강제로 다운그레이드되면서
# Colab에 이미 깔려있는 tensorflow(protobuf>=5.28 요구)가 깨지는 문제가 있었습니다.
# 최신 버전을 쓰면 이 문제가 없습니다.
!pip install -q -U mediapipe tqdm

# torch/torchvision은 Colab에 기본 설치되어 있음 (버전만 확인)
import torch, torchvision
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("torchvision", torchvision.__version__)

import mediapipe as mp
from mediapipe.tasks.python import vision as mp_vision
assert hasattr(mp_vision, "FaceLandmarker"), "mediapipe Tasks API(FaceLandmarker)를 찾을 수 없습니다."
print("mediapipe OK:", mp.__version__)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 15.0 MB/s eta 0:00:00
torch 2.11.0+cu128 | cuda available: True
torchvision 0.26.0+cu128
mediapipe OK: 1.0.1


In [3]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
# 작업 폴더 준비
import os
PROJECT_DIR = "/content/eyebrow_poc"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("작업 디렉토리:", os.getcwd())


작업 디렉토리: /content/eyebrow_poc


In [5]:
%%writefile data_prep.py
"""
data_prep.py
------------
원본 얼굴 이미지 -> (1) 정렬된 얼굴 crop, (2) eyebrow density mask,
(3) front/middle/tail 3분할 region mask, (4) pseudo eyebrow-removed 이미지
를 생성해서 학습에 쓸 수 있는 형태로 저장한다.

mediapipe FaceLandmarker(Tasks API)를 사용한다.

** 버전 관련 중요 참고 **
이전 버전은 legacy `mp.solutions.face_mesh`(mediapipe<=0.10.13)를 사용했는데,
이 API는 오래된 protobuf(<5)를 요구해서 pip 설치 시 protobuf를 강제로
다운그레이드시킨다. Colab에는 이미 protobuf>=5.28을 요구하는 tensorflow가
깔려있어서 이 다운그레이드가 tensorflow import를 깨뜨리고, 그 여파로
mediapipe import 자체도 실패하는 문제가 있었다.
-> 이 버전은 최신 mediapipe(Tasks API, `mediapipe.tasks.python.vision`)만
   사용하고 `mp.solutions`에 전혀 의존하지 않는다. mediapipe 버전을 특정
   숫자로 pin할 필요가 없다 (`pip install mediapipe`만 하면 됨).

Tasks API는 `face_landmarker.task` 모델 파일이 필요하며, 최초 실행 시
자동으로 다운로드한다 (인터넷 연결 필요, Colab에서는 문제없음).

사용 예:
    python data_prep.py --input_dir raw_faces --output_dir processed --role user
    python data_prep.py --input_dir raw_treatment --output_dir processed --role treatment

    # celebrity는 인물별로 폴더가 분리돼 있는 경우가 많으므로(예: 차은우/, 고윤정/),
    # 각 폴더를 따로 --celeb_prefix로 지정해서 실행한다 (같은 output_dir에 누적됨).
    # dataset.py의 EyebrowStyleRefDataset은 파일명 접두사(stem.split("_")[0])로
    # celeb_id를 구분하므로, celeb_prefix 자체에는 '_'를 넣지 않는다.
    python data_prep.py --input_dir "차은우" --output_dir processed --role celeb --celeb_prefix chaeunwoo
    python data_prep.py --input_dir "고윤정" --output_dir processed --role celeb --celeb_prefix gohyunjung

role:
    user       -> 일반 얼굴 (identity/texture 학습용). mask+density+pseudo removal 모두 생성
    celeb      -> celebrity eyebrow reference. eyebrow crop + density만 생성 (removal 불필요)
    treatment  -> 시술 후 사진. density prior 통계 추출용 (crop + density만)
"""
import argparse
import json
import os
import urllib.request

import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

IMG_SIZE = 256

MODEL_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
DEFAULT_MODEL_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "face_landmarker.task")


def ensure_model(model_path=DEFAULT_MODEL_PATH):
    """face_landmarker.task 모델이 없으면 다운로드. (약 4~5MB, 최초 1회만)"""
    if not os.path.exists(model_path):
        print(f"[data_prep] face_landmarker 모델 다운로드 중 -> {model_path}")
        urllib.request.urlretrieve(MODEL_URL, model_path)
        print("[data_prep] 다운로드 완료")
    return model_path


def build_face_landmarker(model_path):
    base_options = mp_python.BaseOptions(model_asset_path=model_path)
    options = mp_vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=mp_vision.RunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=0.5,
    )
    return mp_vision.FaceLandmarker.create_from_options(options)


def _connection_indices(connections):
    """mediapipe Connection 목록 -> landmark index 집합.
    mp.solutions 없이도 mediapipe.tasks.python.vision.FaceLandmarksConnections에서
    동일한 인덱스를 얻을 수 있다 (0.10.13의 mp.solutions.face_mesh 결과와 동일함을 확인함)."""
    return sorted({c.start for c in connections} | {c.end for c in connections})


_LEFT_BROW_IDX = _connection_indices(mp_vision.FaceLandmarksConnections.FACE_LANDMARKS_LEFT_EYEBROW)
_RIGHT_BROW_IDX = _connection_indices(mp_vision.FaceLandmarksConnections.FACE_LANDMARKS_RIGHT_EYEBROW)
_FACE_OVAL_IDX = _connection_indices(mp_vision.FaceLandmarksConnections.FACE_LANDMARKS_FACE_OVAL)


def get_landmarks(image_bgr, landmarker):
    """단일 이미지에서 landmark 좌표(pixel 기준)를 반환. 얼굴 미검출 시 None."""
    h, w = image_bgr.shape[:2]
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB).astype(np.uint8)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_image)
    if not result.face_landmarks:
        return None
    lm = result.face_landmarks[0]
    pts = np.array([[p.x * w, p.y * h] for p in lm], dtype=np.float32)
    return pts


def align_and_crop(image_bgr, landmarks, out_size=IMG_SIZE, margin=1.4):
    """face oval landmark로 bounding box를 잡고 정사각형 crop 후 resize.
    (정밀한 유사도 정렬(회전 보정)이 필요하면 눈 중심 각도로 warpAffine 추가 가능)"""
    oval_pts = landmarks[_FACE_OVAL_IDX]
    x_min, y_min = oval_pts.min(axis=0)
    x_max, y_max = oval_pts.max(axis=0)
    cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
    side = max(x_max - x_min, y_max - y_min) * margin / 2

    h, w = image_bgr.shape[:2]
    x1, y1 = int(max(cx - side, 0)), int(max(cy - side, 0))
    x2, y2 = int(min(cx + side, w)), int(min(cy + side, h))
    crop = image_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return None, None
    scale = out_size / crop.shape[1]
    crop_resized = cv2.resize(crop, (out_size, out_size))

    # crop 좌표계로 landmark 변환
    new_landmarks = landmarks.copy()
    new_landmarks[:, 0] = (landmarks[:, 0] - x1) * scale
    new_landmarks[:, 1] = (landmarks[:, 1] - y1) * (out_size / crop.shape[0])
    return crop_resized, new_landmarks


def eyebrow_density_mask(image_bgr, landmarks, brow_idx, dilate_px=6, feather=9):
    """
    continuous density mask M(x,y) in [0,1] 생성.
    1) landmark convex hull로 눈썹 polygon 확보
    2) polygon을 약간 dilate (주변 피부 맥락 포함)
    3) polygon 내부에서 '어두울수록 짙은 눈썹'이라는 가정으로 grayscale intensity를
       density proxy로 사용 (밝을수록 옅음 -> density 낮음)
    4) 경계를 feather(가우시안 블러)해서 continuous mask로 만듦

    -> 이것은 hair-strand 레벨 density 추정기의 근사치(proxy)이며,
       본 논문 설계 문서 7절의 "6번(구간별 강도)~5번(hair-density 기반)" 사이의
       실용적 절충안이다. 정밀한 strand-level 추정을 원하면 hair segmentation
       모델(예: MODNet 계열, hair matting)로 교체 가능.
    """
    h, w = image_bgr.shape[:2]
    pts = landmarks[brow_idx].astype(np.int32)
    hull = cv2.convexHull(pts)

    poly_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(poly_mask, hull, 255)
    if dilate_px > 0:
        kernel = np.ones((dilate_px, dilate_px), np.uint8)
        poly_mask = cv2.dilate(poly_mask, kernel)

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    # 눈썹 영역 내부의 로컬 피부톤(주변부) 대비 어두운 정도를 density로 사용
    region = gray[poly_mask > 0]
    if region.size == 0:
        return np.zeros((h, w), dtype=np.float32)
    skin_ref = np.percentile(region, 80)  # 영역 내 밝은 픽셀 = 피부에 가까움
    density = np.clip((skin_ref - gray) / (skin_ref + 1e-6), 0, 1)
    density = density * (poly_mask.astype(np.float32) / 255.0)

    if feather > 0:
        density = cv2.GaussianBlur(density, (feather * 2 + 1, feather * 2 + 1), 0)
    return density.astype(np.float32)


def region_split_mask(image_shape, landmarks, brow_idx, n_regions=3):
    """눈썹 영역을 x좌표 기준 front/middle/tail 3구간으로 나눈 (n_regions, H, W) mask.
    좌우 눈썹이 섞이지 않도록 이 함수는 한쪽 눈썹 landmark만 받는다."""
    h, w = image_shape[:2]
    pts = landmarks[brow_idx]
    x_min, x_max = pts[:, 0].min(), pts[:, 0].max()
    hull = cv2.convexHull(pts.astype(np.int32))
    full_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(full_mask, hull, 1)

    thirds = np.linspace(x_min, x_max, n_regions + 1)
    out = np.zeros((n_regions, h, w), dtype=np.float32)
    xs = np.arange(w)[None, :].repeat(h, axis=0)
    for i in range(n_regions):
        band = (xs >= thirds[i]) & (xs < thirds[i + 1] + 1)
        out[i] = full_mask * band
    return out


def pseudo_remove_eyebrow(image_bgr, density_mask, inpaint_radius=7):
    """cv2 inpainting으로 눈썹을 제거한 pseudo 'eyebrow-free' 이미지를 생성.
    이 이미지는 실제 시술 전/후 paired data가 없는 상황에서
    (a) 논문의 image-specific direction d_t 계산,
    (b) restoration(backward) 학습의 self-target
    으로 사용하기 위한 self-supervised proxy 이다. 실제 ground truth가 아님에 주의."""
    mask_u8 = (density_mask > 0.15).astype(np.uint8) * 255
    if mask_u8.sum() == 0:
        return image_bgr.copy()
    removed = cv2.inpaint(image_bgr, mask_u8, inpaint_radius, cv2.INPAINT_TELEA)
    return removed


def process_folder(input_dir, output_dir, role, celeb_prefix=None):
    os.makedirs(output_dir, exist_ok=True)
    img_out = os.path.join(output_dir, "images")
    dens_out = os.path.join(output_dir, "density")
    region_out = os.path.join(output_dir, "regions")
    removed_out = os.path.join(output_dir, "pseudo_removed")
    brow_crop_out = os.path.join(output_dir, "brow_crops")
    for d in [img_out, dens_out, region_out, removed_out, brow_crop_out]:
        os.makedirs(d, exist_ok=True)

    manifest = []
    model_path = ensure_model()
    landmarker = build_face_landmarker(model_path)
    try:
        for fname in sorted(os.listdir(input_dir)):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png",".avif", ".webp", ".bmp")):
                print(f"[skip] 지원하지 않는 확장자: {fname}")
                continue
            path = os.path.join(input_dir, fname)
            img = cv2.imread(path)
            if img is None:
                print(f"[skip] cannot read {fname}")
                continue

            landmarks = get_landmarks(img, landmarker)
            if landmarks is None:
                print(f"[skip] no face detected: {fname}")
                continue

            aligned, aligned_lm = align_and_crop(img, landmarks)
            if aligned is None:
                print(f"[skip] bad crop: {fname}")
                continue

            dens_left = eyebrow_density_mask(aligned, aligned_lm, _LEFT_BROW_IDX)
            dens_right = eyebrow_density_mask(aligned, aligned_lm, _RIGHT_BROW_IDX)
            density = np.maximum(dens_left, dens_right)

            region_left = region_split_mask(aligned.shape, aligned_lm, _LEFT_BROW_IDX)
            region_right = region_split_mask(aligned.shape, aligned_lm, _RIGHT_BROW_IDX)
            region = np.maximum(region_left, region_right)  # (3, H, W)

            raw_stem = os.path.splitext(fname)[0]
            stem = f"{celeb_prefix}_{raw_stem}" if celeb_prefix else raw_stem
            cv2.imwrite(os.path.join(img_out, f"{stem}.png"), aligned)
            np.save(os.path.join(dens_out, f"{stem}.npy"), density)
            np.save(os.path.join(region_out, f"{stem}.npy"), region)

            # eyebrow-only crop (celebrity style extractor 입력용) - 양쪽 눈썹 합친 bbox
            all_idx = _LEFT_BROW_IDX + _RIGHT_BROW_IDX
            pts = aligned_lm[all_idx].astype(np.int32)
            x1, y1 = pts.min(axis=0)
            x2, y2 = pts.max(axis=0)
            pad = 15
            h, w = aligned.shape[:2]
            x1, y1 = max(x1 - pad, 0), max(y1 - pad, 0)
            x2, y2 = min(x2 + pad, w), min(y2 + pad, h)
            brow_crop = aligned[y1:y2, x1:x2]
            if brow_crop.size > 0:
                brow_crop = cv2.resize(brow_crop, (128, 64))
                cv2.imwrite(os.path.join(brow_crop_out, f"{stem}.png"), brow_crop)

            entry = {"stem": stem, "role": role}

            if role in ("user",):
                removed = pseudo_remove_eyebrow(aligned, density)
                cv2.imwrite(os.path.join(removed_out, f"{stem}.png"), removed)
                entry["has_pseudo_removed"] = True

            manifest.append(entry)
            print(f"[ok] {fname}")
    finally:
        landmarker.close()

    manifest_path = os.path.join(output_dir, f"manifest_{role}.json")
    if os.path.exists(manifest_path):
        # celeb role처럼 같은 output_dir에 여러 번(인물별 폴더) 나눠서 실행하는 경우
        # 기존 manifest에 이어붙인다 (stem 기준 중복은 덮어쓰기).
        with open(manifest_path) as f:
            existing = json.load(f)
        merged = {e["stem"]: e for e in existing}
        merged.update({e["stem"]: e for e in manifest})
        manifest = list(merged.values())

    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    print(f"\n{role}: 총 누적 {len(manifest)}개 -> {manifest_path}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_dir", required=True)
    parser.add_argument("--output_dir", required=True)
    parser.add_argument("--role", required=True, choices=["user", "celeb", "treatment"])
    parser.add_argument("--celeb_prefix", default=None,
                         help="celeb role일 때 파일명 앞에 붙일 celeb_id (예: chaeunwoo). '_' 포함 금지.")
    args = parser.parse_args()
    if args.celeb_prefix and "_" in args.celeb_prefix:
        raise ValueError("celeb_prefix에는 '_'를 넣지 마세요 (celeb_id 파싱이 stem.split('_')[0] 방식입니다).")
    process_folder(args.input_dir, args.output_dir, args.role, celeb_prefix=args.celeb_prefix)


Writing data_prep.py


In [6]:
%%writefile dataset.py
"""
dataset.py
----------
data_prep.py로 만든 processed/ 폴더를 읽어오는 PyTorch Dataset.

processed/
  images/<stem>.png            (256x256 aligned face)
  density/<stem>.npy            (H, W) float32, 0~1
  regions/<stem>.npy            (3, H, W) float32, front/middle/tail
  pseudo_removed/<stem>.png     (user role만 존재)
  brow_crops/<stem>.png         (128x64 eyebrow-only crop, style extractor 입력)
  manifest_<role>.json
"""
import json
import os

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

IMG_SIZE = 256

_img_tf = transforms.Compose([
    transforms.ToTensor(),                       # [0,1]
    transforms.Normalize([0.5] * 3, [0.5] * 3),   # [-1,1]
])

_brow_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])


class EyebrowUserDataset(Dataset):
    """Experiment 1 (removal) + Experiment 2의 self-restoration 학습에 사용.
    반환: original, pseudo_removed, density_mask, region_mask, self_brow_crop
    """

    def __init__(self, root):
        self.root = root
        manifest_path = os.path.join(root, "manifest_user.json")
        with open(manifest_path) as f:
            self.entries = [e for e in json.load(f) if e.get("has_pseudo_removed")]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        stem = self.entries[idx]["stem"]
        img = Image.open(os.path.join(self.root, "images", f"{stem}.png")).convert("RGB")
        removed = Image.open(os.path.join(self.root, "pseudo_removed", f"{stem}.png")).convert("RGB")
        brow = Image.open(os.path.join(self.root, "brow_crops", f"{stem}.png")).convert("RGB")
        density = np.load(os.path.join(self.root, "density", f"{stem}.npy"))
        region = np.load(os.path.join(self.root, "regions", f"{stem}.npy"))

        return {
            "stem": stem,
            "image": _img_tf(img),
            "pseudo_removed": _img_tf(removed),
            "density": torch.from_numpy(density).unsqueeze(0).float(),   # (1,H,W)
            "region": torch.from_numpy(region).float(),                  # (3,H,W)
            "self_brow_crop": _brow_tf(brow),
        }


class EyebrowStyleRefDataset(Dataset):
    """Experiment 2에서 celebrity style reference로 사용할 eyebrow crop 목록.
    celeb_id는 파일명 접두사(예: celebA_01.png -> 'celebA')로 그룹핑한다고 가정.
    """

    def __init__(self, root):
        self.root = root
        manifest_path = os.path.join(root, "manifest_celeb.json")
        with open(manifest_path) as f:
            self.entries = json.load(f)

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        stem = self.entries[idx]["stem"]
        brow = Image.open(os.path.join(self.root, "brow_crops", f"{stem}.png")).convert("RGB")
        celeb_id = stem.split("_")[0]
        return {"stem": stem, "celeb_id": celeb_id, "brow_crop": _brow_tf(brow)}

    def get_by_celeb_id(self, celeb_id):
        idxs = [i for i, e in enumerate(self.entries) if e["stem"].split("_")[0] == celeb_id]
        return idxs


Writing dataset.py


In [7]:
%%writefile model.py
"""
model.py
--------
Experiment 1 (attribute removal) / Experiment 2 (style manipulation)를 실제로
돌려볼 수 있는 경량 U-Net 스타일 오토인코더.

큰 diffusion backbone 대신, 150~250장 규모의 소규모 데이터로도 학습 가능한
conv encoder-decoder를 사용한다. (연구 설계 문서 Phase 1~6에 해당하는 PoC 버전.
Phase 7~8에서 더 강력한 decoder/backbone으로 교체 가능)

핵심 아이디어:
  - skip connection에 (1 - density_mask)를 곱해서 피부/모공 texture는
    그대로 전달하되, 눈썹 자체의 정보는 skip으로 새지 않도록 막는다.
  - 눈썹 정보는 bottleneck에서 density-aware removal + style modulation을
    거쳐서만 복원되도록 강제한다 -> identity/texture는 skip으로, eyebrow
    style은 style code로 분리해서 흐르게 하는 구조적 disentanglement.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def conv_block(c_in, c_out, stride=2):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 4, stride=stride, padding=1),
        nn.GroupNorm(min(8, c_out), c_out),
        nn.SiLU(),
    )


def up_block(c_in, c_out):
    return nn.Sequential(
        nn.Upsample(scale_factor=2, mode="nearest"),
        nn.Conv2d(c_in, c_out, 3, padding=1),
        nn.GroupNorm(min(8, c_out), c_out),
        nn.SiLU(),
    )


class Encoder(nn.Module):
    """256 -> 128 -> 64 -> 32 -> 16, channel 32->64->128->256->512"""

    def __init__(self, in_ch=3, chs=(32, 64, 128, 256, 512)):
        super().__init__()
        self.stem = nn.Conv2d(in_ch, chs[0], 3, padding=1)
        self.downs = nn.ModuleList([
            conv_block(chs[i], chs[i + 1]) for i in range(len(chs) - 1)
        ])

    def forward(self, x):
        feats = []
        f = self.stem(x)
        feats.append(f)  # 256
        for down in self.downs:
            f = down(f)
            feats.append(f)
        return feats  # [256,128,64,32,16] 해상도 각 스케일 feature (마지막이 bottleneck)


class DensityAwareRemoval(nn.Module):
    """bottleneck feature에서 density mask를 이용해 eyebrow attribute를 제거.
    density_mask는 원 해상도(256) 기준이므로 bottleneck 해상도로 downsample해서 사용."""

    def __init__(self, feat_ch):
        super().__init__()
        self.direction_net = nn.Sequential(
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(feat_ch, feat_ch, 3, padding=1),
        )

    def forward(self, f_bottleneck, density_mask):
        _, _, h, w = f_bottleneck.shape
        m = F.interpolate(density_mask, size=(h, w), mode="bilinear", align_corners=False)
        d_t = self.direction_net(f_bottleneck)
        f_removed = f_bottleneck - m * d_t
        return f_removed, d_t


class StyleExtractor(nn.Module):
    """eyebrow crop(128x64)에서 style code 추출. celebrity/self 공용."""

    def __init__(self, style_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            conv_block(3, 32, stride=2),    # 64x32
            conv_block(32, 64, stride=2),   # 32x16
            conv_block(64, 128, stride=2),  # 16x8
            conv_block(128, 256, stride=2), # 8x4
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(256, style_dim)

    def forward(self, brow_crop):
        f = self.net(brow_crop)
        f = self.pool(f).flatten(1)
        return self.proj(f)


class HierarchicalStyleModulation(nn.Module):
    """논문의 hierarchical(tag->attribute) 설계를
    front/middle/tail 3-region 계층으로 재정의.
    각 region마다 별도의 learnable vector + AdaIN, region mask로 가중합."""

    def __init__(self, feat_ch, style_dim, n_regions=3):
        super().__init__()
        self.n_regions = n_regions
        self.learnable_vecs = nn.Parameter(torch.randn(n_regions, feat_ch) * 0.02)
        self.style_proj = nn.ModuleList([
            nn.Linear(style_dim, feat_ch * 2) for _ in range(n_regions)
        ])
        self.cross_attn = nn.MultiheadAttention(feat_ch, num_heads=4, batch_first=True)
        self.out_conv = nn.Conv2d(feat_ch, feat_ch, 3, padding=1)

    def _adain(self, f, style, region_idx):
        scale, shift = self.style_proj[region_idx](style).chunk(2, dim=-1)
        lv = self.learnable_vecs[region_idx].view(1, -1, 1, 1)
        mean = f.mean(dim=[2, 3], keepdim=True)
        std = f.std(dim=[2, 3], keepdim=True) + 1e-5
        f_norm = (f - mean) / std + lv * 0.1
        return f_norm * scale.unsqueeze(-1).unsqueeze(-1) + shift.unsqueeze(-1).unsqueeze(-1)

    def forward(self, f_removed, style_code, region_mask):
        """
        f_removed: (B,C,h,w) bottleneck feature (removal 이후)
        style_code: (B, style_dim)
        region_mask: (B,3,H,W) 원본 해상도 -> bottleneck 해상도로 resize해서 사용
        """
        b, c, h, w = f_removed.shape
        region_small = F.interpolate(region_mask, size=(h, w), mode="bilinear", align_corners=False)
        region_small = region_small / (region_small.sum(dim=1, keepdim=True) + 1e-6)  # 정규화(가중합)

        out = 0
        for r in range(self.n_regions):
            modulated = self._adain(f_removed, style_code, r)
            out = out + modulated * region_small[:, r:r + 1]

        b_, c_, h_, w_ = out.shape
        flat = out.flatten(2).transpose(1, 2)
        attn_out, _ = self.cross_attn(flat, flat, flat)
        out = attn_out.transpose(1, 2).reshape(b_, c_, h_, w_)
        return self.out_conv(out)


class Decoder(nn.Module):
    """bottleneck(16) -> 256, skip connection에 (1-density) gating 적용."""

    def __init__(self, chs=(512, 256, 128, 64, 32)):
        super().__init__()
        self.ups = nn.ModuleList([
            up_block(chs[i] * 2 if i > 0 else chs[i], chs[i + 1]) for i in range(len(chs) - 1)
        ])
        self.out_conv = nn.Conv2d(chs[-1] * 2, 3, 3, padding=1)

    def forward(self, f_bottleneck, enc_feats, density_mask):
        """
        enc_feats: encoder가 반환한 [f256, f128, f64, f32, f16] 중 skip에 쓸
                   [f256, f128, f64, f32] (bottleneck 제외, 얕은 순서대로)
        density_mask: (B,1,256,256)
        """
        f = f_bottleneck
        skips = enc_feats[:-1][::-1]  # [f32, f64, f128, f256] 순서 (깊은 데서 얕은 순)
        for i, up in enumerate(self.ups):
            f = up(f)
            skip = skips[i]
            _, _, h, w = skip.shape
            gate = 1.0 - F.interpolate(density_mask, size=(h, w), mode="bilinear", align_corners=False)
            f = torch.cat([f, skip * gate], dim=1)
        out = self.out_conv(f)
        return torch.tanh(out)


class EyebrowManipulator(nn.Module):
    """Experiment 1 & 2에 공용으로 쓰는 전체 모델."""

    def __init__(self, style_dim=256):
        super().__init__()
        self.encoder = Encoder()
        self.removal = DensityAwareRemoval(feat_ch=512)
        self.style_extractor = StyleExtractor(style_dim=style_dim)
        self.style_mod = HierarchicalStyleModulation(feat_ch=512, style_dim=style_dim)
        self.decoder = Decoder()

    def encode(self, image):
        feats = self.encoder(image)
        return feats  # feats[-1] = bottleneck (16x16, 512ch)

    def remove_and_restyle(self, feats, density_mask, region_mask, style_code):
        f_removed, d_t = self.removal(feats[-1], density_mask)
        f_styled = self.style_mod(f_removed, style_code, region_mask)
        out_img = self.decoder(f_styled, feats, density_mask)
        return out_img, f_removed, d_t

    def forward(self, image, density_mask, region_mask, ref_brow_crop):
        """편의용 end-to-end forward: self image + 임의 reference eyebrow crop"""
        feats = self.encode(image)
        style_code = self.style_extractor(ref_brow_crop)
        out_img, f_removed, d_t = self.remove_and_restyle(feats, density_mask, region_mask, style_code)
        return out_img, f_removed, d_t


if __name__ == "__main__":
    # 빠른 shape 체크 (실제 학습 없이 forward만)
    model = EyebrowManipulator()
    img = torch.randn(2, 3, 256, 256)
    dens = torch.rand(2, 1, 256, 256)
    region = torch.rand(2, 3, 256, 256)
    brow = torch.randn(2, 3, 64, 128)
    out, f_rm, d_t = model(img, dens, region, brow)
    print("output:", out.shape)          # (2,3,256,256)
    print("removed feat:", f_rm.shape)   # (2,512,16,16)


Writing model.py


In [8]:
%%writefile train_exp1_removal.py
"""
train_exp1_removal.py
----------------------
Experiment 1 (Attribute Removal) 학습.

목표: 원본 -> eyebrow-removed 이미지를 생성하되
      (a) 눈썹만 제거되고 (b) 피부 texture/모공이 보존되고 (c) identity가 유지되는가.

학습 신호(모두 self/pseudo-supervised, paired 시술 데이터 불필요):
  1) reconstruction: self style code로 restoration -> 원본과 최대한 같아야 함
     (논문의 forward-backward consistency, 식(7) L_perc에 대응)
  2) removal 방향성: pseudo_removed(inpainting) 이미지와 removal 결과가
     눈썹 영역에서는 가까워지고, 눈썹 외 영역에서는 원본과 가까워야 함
  3) identity: 간단한 low-level identity proxy로 얼굴 하관/눈 주변 patch L1
     (본격적인 face-recognition embedding은 evaluate.py에서 별도로 사용 권장)

사용:
    python train_exp1_removal.py --data_root processed --epochs 50 --out_dir ckpt_exp1
"""
import argparse
import os

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset import EyebrowUserDataset
from model import EyebrowManipulator


def texture_preservation_loss(pred, target, density_mask):
    """눈썹 외(피부) 영역에서의 L1 — 모공/피부결 보존을 명시적으로 감독."""
    weight = 1.0 - density_mask  # 눈썹 아닌 곳일수록 가중치 높음
    return (weight * (pred - target).abs()).mean()


def removal_region_loss(pred, pseudo_removed, density_mask):
    """눈썹 영역에서는 pseudo-removed(inpainted) 결과에 가까워지도록."""
    weight = density_mask
    return (weight * (pred - pseudo_removed).abs()).mean()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_root", required=True)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--out_dir", default="ckpt_exp1")
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    device = torch.device(args.device)

    ds = EyebrowUserDataset(args.data_root)
    print(f"학습 이미지 수: {len(ds)}")
    dl = DataLoader(ds, batch_size=args.batch_size, shuffle=True, num_workers=2, drop_last=True)

    model = EyebrowManipulator().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)

    for epoch in range(args.epochs):
        model.train()
        total_loss = 0.0
        pbar = tqdm(dl, desc=f"epoch {epoch+1}/{args.epochs}")
        for batch in pbar:
            image = batch["image"].to(device)
            pseudo_removed = batch["pseudo_removed"].to(device)
            density = batch["density"].to(device)
            region = batch["region"].to(device)
            self_brow = batch["self_brow_crop"].to(device)

            feats = model.encode(image)

            # ---- (A) self-restoration: 자기 자신의 style로 원본을 복원 ----
            self_style = model.style_extractor(self_brow)
            restored, _, _ = model.remove_and_restyle(feats, density, region, self_style)
            loss_recon = F.l1_loss(restored, image)
            loss_texture = texture_preservation_loss(restored, image, density)

            # ---- (B) removal 방향 확인: style code를 0벡터(=無스타일)로 주고
            #          눈썹 영역이 pseudo_removed 쪽으로 가는지 감독 ----
            zero_style = torch.zeros_like(self_style)
            removed_out, f_removed, d_t = model.remove_and_restyle(feats, density, region, zero_style)
            loss_removal = removal_region_loss(removed_out, pseudo_removed, density)
            loss_removal_texture = texture_preservation_loss(removed_out, image, density)

            loss = (1.0 * loss_recon + 0.5 * loss_texture
                    + 1.0 * loss_removal + 0.5 * loss_removal_texture)

            opt.zero_grad()
            loss.backward()
            opt.step()

            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())

        avg = total_loss / len(dl)
        print(f"[epoch {epoch+1}] avg_loss={avg:.4f}")

        if (epoch + 1) % 10 == 0 or epoch == args.epochs - 1:
            ckpt_path = os.path.join(args.out_dir, f"model_epoch{epoch+1}.pt")
            torch.save(model.state_dict(), ckpt_path)
            print(f"저장: {ckpt_path}")


if __name__ == "__main__":
    main()


Writing train_exp1_removal.py


In [9]:
%%writefile train_exp2_style.py
"""
train_exp2_style.py
--------------------
Experiment 2 (Style Manipulation) 학습.
train_exp1_removal.py 로 만든 checkpoint를 이어받아 style modulation을 정교화한다.

우리에게는 "celebrity 스타일 A/B/C를 적용한 정답 이미지"가 없다 (paired 데이터 없음).
그래서 논문의 forward-backward consistency 철학을 그대로 계승하되, shape ground
truth 없이도 학습 가능한 2가지 self-supervised 신호를 추가한다.

  1) Style-code cycle consistency (신규, 논문에 없음):
     생성된 이미지에서 다시 눈썹 crop을 떼어 style_extractor에 통과시키면
     처음 넣어준 celebrity style code와 비슷해야 한다.
     -> "스타일이 실제로 반영됐는지"를 감독하는 대체 shape loss 역할.

  2) Identity/texture 보존 (exp1과 동일 loss 계승):
     눈썹 영역 밖은 원본과 동일해야 한다 (density mask로 가중).

  3) Style discriminability (contrastive):
     서로 다른 celebrity의 style code는 embedding space에서 떨어져 있어야
     스타일 A/B/C가 실제로 구분되는 결과를 만든다 (Experiment 2 disentanglement 검증에 필요).

사용:
    python train_exp2_style.py --data_root processed --init_ckpt ckpt_exp1/model_epoch50.pt \
        --epochs 30 --out_dir ckpt_exp2
"""
import argparse
import os
import random

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from dataset import EyebrowUserDataset, EyebrowStyleRefDataset
from model import EyebrowManipulator


def texture_preservation_loss(pred, target, density_mask):
    weight = 1.0 - density_mask
    return (weight * (pred - target).abs()).mean()


def style_cycle_loss(model, out_img, region_mask, target_style_code, device):
    """생성 이미지에서 눈썹 부분을 대략적으로 다시 crop(bbox 근사)해서
    style_extractor에 통과시키고, 원래 준 style code와 cosine 거리로 비교.

    region_mask(front/middle/tail 합)로 눈썹 영역의 bounding box를 근사한다.
    """
    b, _, h, w = out_img.shape
    brow_mask = region_mask.sum(dim=1, keepdim=True)  # (B,1,H,W)
    crops = []
    for i in range(b):
        ys, xs = torch.where(brow_mask[i, 0] > 0.05)
        if len(xs) == 0:
            crops.append(F.interpolate(out_img[i:i + 1], size=(64, 128)))
            continue
        x1, x2 = xs.min().item(), xs.max().item()
        y1, y2 = ys.min().item(), ys.max().item()
        pad = 10
        x1, y1 = max(x1 - pad, 0), max(y1 - pad, 0)
        x2, y2 = min(x2 + pad, w - 1), min(y2 + pad, h - 1)
        crop = out_img[i:i + 1, :, y1:y2 + 1, x1:x2 + 1]
        crops.append(F.interpolate(crop, size=(64, 128), mode="bilinear", align_corners=False))
    crops = torch.cat(crops, dim=0)

    re_style = model.style_extractor(crops)
    cos = F.cosine_similarity(re_style, target_style_code, dim=-1)
    return (1 - cos).mean()


def contrastive_style_loss(style_codes, celeb_ids, margin=0.3):
    """같은 celeb -> 가깝게, 다른 celeb -> margin 이상 멀게 (배치 내 pairwise)."""
    b = style_codes.size(0)
    sim = F.cosine_similarity(style_codes.unsqueeze(1), style_codes.unsqueeze(0), dim=-1)
    same = torch.tensor([[1.0 if celeb_ids[i] == celeb_ids[j] else 0.0
                           for j in range(b)] for i in range(b)], device=style_codes.device)
    pos_loss = ((1 - sim) * same).sum() / (same.sum() + 1e-6)
    neg = 1 - same
    neg_loss = (F.relu(sim - margin) * neg).sum() / (neg.sum() + 1e-6)
    return pos_loss + neg_loss


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_root", required=True)
    parser.add_argument("--init_ckpt", default=None)
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--batch_size", type=int, default=8)
    parser.add_argument("--lr", type=float, default=5e-5)
    parser.add_argument("--out_dir", default="ckpt_exp2")
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    device = torch.device(args.device)

    user_ds = EyebrowUserDataset(args.data_root)
    style_ds = EyebrowStyleRefDataset(args.data_root)
    print(f"user 이미지: {len(user_ds)}, celeb 눈썹 reference: {len(style_ds)}")

    dl = DataLoader(user_ds, batch_size=args.batch_size, shuffle=True, num_workers=2, drop_last=True)

    model = EyebrowManipulator().to(device)
    if args.init_ckpt:
        model.load_state_dict(torch.load(args.init_ckpt, map_location=device))
        print(f"초기화: {args.init_ckpt}")

    opt = torch.optim.Adam(model.parameters(), lr=args.lr)

    for epoch in range(args.epochs):
        model.train()
        total = 0.0
        pbar = tqdm(dl, desc=f"epoch {epoch+1}/{args.epochs}")
        for batch in pbar:
            image = batch["image"].to(device)
            density = batch["density"].to(device)
            region = batch["region"].to(device)
            b = image.size(0)

            # 배치마다 랜덤 celebrity reference 샘플링
            idxs = random.sample(range(len(style_ds)), b)
            style_batch = [style_ds[i] for i in idxs]
            brow_crops = torch.stack([s["brow_crop"] for s in style_batch]).to(device)
            celeb_ids = [s["celeb_id"] for s in style_batch]

            feats = model.encode(image)
            style_codes = model.style_extractor(brow_crops)

            out_img, f_removed, _ = model.remove_and_restyle(feats, density, region, style_codes)

            loss_texture = texture_preservation_loss(out_img, image, density)
            loss_cycle = style_cycle_loss(model, out_img, region, style_codes, device)
            loss_contrastive = contrastive_style_loss(style_codes, celeb_ids)

            loss = 1.0 * loss_texture + 1.0 * loss_cycle + 0.3 * loss_contrastive

            opt.zero_grad()
            loss.backward()
            opt.step()

            total += loss.item()
            pbar.set_postfix(loss=loss.item(), tex=loss_texture.item(), cyc=loss_cycle.item())

        print(f"[epoch {epoch+1}] avg_loss={total/len(dl):.4f}")
        if (epoch + 1) % 10 == 0 or epoch == args.epochs - 1:
            ckpt_path = os.path.join(args.out_dir, f"model_epoch{epoch+1}.pt")
            torch.save(model.state_dict(), ckpt_path)
            print(f"저장: {ckpt_path}")


if __name__ == "__main__":
    main()


Writing train_exp2_style.py


In [10]:
%%writefile run_experiments.py
"""
run_experiments.py
-------------------
학습된 checkpoint로 Experiment 1(removal), Experiment 2(style manipulation)를
직접 눈으로 확인할 수 있도록 결과 이미지를 저장한다.

사용:
    # Experiment 1: 특정 user 이미지의 removal 결과
    python run_experiments.py exp1 --ckpt ckpt_exp1/model_epoch50.pt \
        --data_root processed --stem user_0001 --out_dir results/exp1

    # Experiment 2: 한 user에게 여러 celebrity style 적용
    python run_experiments.py exp2 --ckpt ckpt_exp2/model_epoch30.pt \
        --data_root processed --stem user_0001 \
        --celeb_ids celebA celebB celebC --out_dir results/exp2
"""
import argparse
import json
import os

import torch
import torchvision.utils as vutils
from PIL import Image
from torchvision import transforms

from dataset import EyebrowStyleRefDataset, EyebrowUserDataset, _brow_tf, _img_tf
from model import EyebrowManipulator

_to_pil = transforms.Compose([
    transforms.Normalize([-1, -1, -1], [2, 2, 2]),  # [-1,1] -> [0,1]
])


def load_model(ckpt, device):
    model = EyebrowManipulator().to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()
    return model


def save_grid(tensors, path, nrow):
    imgs = torch.stack([_to_pil(t.clamp(-1, 1)) for t in tensors])
    vutils.save_image(imgs, path, nrow=nrow)


def exp1_removal(args, device):
    """Experiment 1: original / pseudo_removed(참고용) / model removal 결과 비교."""
    ds = EyebrowUserDataset(args.data_root)
    stem_to_idx = {e["stem"]: i for i, e in enumerate(ds.entries)}
    if args.stem not in stem_to_idx:
        raise ValueError(f"{args.stem} not found in user manifest")
    item = ds[stem_to_idx[args.stem]]

    model = load_model(args.ckpt, device)
    image = item["image"].unsqueeze(0).to(device)
    density = item["density"].unsqueeze(0).to(device)
    region = item["region"].unsqueeze(0).to(device)
    pseudo = item["pseudo_removed"].unsqueeze(0).to(device)

    with torch.no_grad():
        feats = model.encode(image)
        zero_style = torch.zeros(1, 256, device=device)
        removed_out, f_removed, d_t = model.remove_and_restyle(feats, density, region, zero_style)

    os.makedirs(args.out_dir, exist_ok=True)
    save_grid([image[0], pseudo[0], removed_out[0]],
              os.path.join(args.out_dir, f"{args.stem}_exp1_compare.png"), nrow=3)
    print(f"저장 완료: {args.out_dir}/{args.stem}_exp1_compare.png (원본 | pseudo-removed 참고 | 모델 removal)")

    # 정량 체크: 눈썹 외 영역 L1 (identity/texture 보존), 눈썹 영역 변화량
    outside_l1 = ((1 - density) * (removed_out - image).abs()).mean().item()
    inside_change = (density * (removed_out - image).abs()).mean().item()
    print(f"[Exp1 지표] 눈썹 외 L1(작을수록 identity/texture 보존 good)={outside_l1:.4f}")
    print(f"[Exp1 지표] 눈썹 영역 변화량(removal 발생 여부, 클수록 removal 효과 큼)={inside_change:.4f}")


def exp2_style(args, device):
    """Experiment 2: 같은 user에게 여러 celebrity style 적용 -> 얼굴/피부는 유지, 눈썹만 변화하는지."""
    user_ds = EyebrowUserDataset(args.data_root)
    style_ds = EyebrowStyleRefDataset(args.data_root)
    stem_to_idx = {e["stem"]: i for i, e in enumerate(user_ds.entries)}
    if args.stem not in stem_to_idx:
        raise ValueError(f"{args.stem} not found in user manifest")
    item = user_ds[stem_to_idx[args.stem]]

    model = load_model(args.ckpt, device)
    image = item["image"].unsqueeze(0).to(device)
    density = item["density"].unsqueeze(0).to(device)
    region = item["region"].unsqueeze(0).to(device)

    with torch.no_grad():
        feats = model.encode(image)

    outputs = [image[0]]
    labels = ["original"]
    for celeb_id in args.celeb_ids:
        idxs = style_ds.get_by_celeb_id(celeb_id)
        if not idxs:
            print(f"[경고] celeb_id={celeb_id} 에 해당하는 눈썹 reference 없음, 건너뜀")
            continue
        brow_crop = style_ds[idxs[0]]["brow_crop"].unsqueeze(0).to(device)
        with torch.no_grad():
            style_code = model.style_extractor(brow_crop)
            out_img, _, _ = model.remove_and_restyle(feats, density, region, style_code)
        outputs.append(out_img[0])
        labels.append(celeb_id)

    os.makedirs(args.out_dir, exist_ok=True)
    save_grid(outputs, os.path.join(args.out_dir, f"{args.stem}_exp2_styles.png"), nrow=len(outputs))
    print(f"저장 완료: {args.out_dir}/{args.stem}_exp2_styles.png  순서: {labels}")

    # disentanglement 정량 체크: 서로 다른 style 결과 간 '눈썹 외' 영역이 서로 비슷한가
    outside_diffs = []
    for i in range(1, len(outputs)):
        for j in range(i + 1, len(outputs)):
            d = ((1 - density[0]) * (outputs[i] - outputs[j]).abs()).mean().item()
            outside_diffs.append(d)
    if outside_diffs:
        avg = sum(outside_diffs) / len(outside_diffs)
        print(f"[Exp2 지표] 서로 다른 style 간 '눈썹 외' 영역 평균 L1 차이"
              f"(작을수록 identity/texture가 잘 보존된 것)={avg:.4f}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    sub = parser.add_subparsers(dest="cmd", required=True)

    p1 = sub.add_parser("exp1")
    p1.add_argument("--ckpt", required=True)
    p1.add_argument("--data_root", required=True)
    p1.add_argument("--stem", required=True)
    p1.add_argument("--out_dir", default="results/exp1")

    p2 = sub.add_parser("exp2")
    p2.add_argument("--ckpt", required=True)
    p2.add_argument("--data_root", required=True)
    p2.add_argument("--stem", required=True)
    p2.add_argument("--celeb_ids", nargs="+", required=True)
    p2.add_argument("--out_dir", default="results/exp2")

    args = parser.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if args.cmd == "exp1":
        exp1_removal(args, device)
    elif args.cmd == "exp2":
        exp2_style(args, device)


Writing run_experiments.py


In [11]:
GENERAL_FACE_DIR = "/content/drive/MyDrive/GeneralFace"
CHAEUNWOO_DIR = "/content/drive/MyDrive/차은우"
GOHYUNJUNG_DIR = "/content/drive/MyDrive/고윤정"

for p in [GENERAL_FACE_DIR, CHAEUNWOO_DIR, GOHYUNJUNG_DIR]:
    exists = os.path.isdir(p)
    n_files = len(os.listdir(p)) if exists else 0
    print(f"{p} -> 존재={exists}, 파일수={n_files}")


/content/drive/MyDrive/GeneralFace -> 존재=False, 파일수=0
/content/drive/MyDrive/차은우 -> 존재=False, 파일수=0
/content/drive/MyDrive/고윤정 -> 존재=False, 파일수=0


In [21]:
PROCESSED_DIR = "/content/drive/MyDrive/eyebrow_poc_processed"  # 전처리 결과 저장 위치 (Drive에 보존)
os.makedirs(PROCESSED_DIR, exist_ok=True)

!python data_prep.py --input_dir "{GENERAL_FACE_DIR}" --output_dir "{PROCESSED_DIR}" --role user


W0000 00:00:1787060686.744252    6585 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787060686.773191    6590 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787060686.792147    6591 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Traceback (most recent call last):
  File "/content/eyebrow_poc/data_prep.py", line 295, in <module>
    process_folder(args.input_dir, args.output_dir, args.role, celeb_prefix=args.celeb_prefix)
  File "/content/eyebrow_poc/data_prep.py", line 210, in process_folder
    for fname in sorted(os.listdir(input_dir)):
                        ^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/GeneralF

In [13]:
!python data_prep.py --input_dir "{CHAEUNWOO_DIR}" --output_dir "{PROCESSED_DIR}" --role celeb --celeb_prefix chaeunwoo


W0000 00:00:1787060533.472155    5675 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787060533.493157    5680 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787060533.512031    5679 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Traceback (most recent call last):
  File "/content/eyebrow_poc/data_prep.py", line 295, in <module>
    process_folder(args.input_dir, args.output_dir, args.role, celeb_prefix=args.celeb_prefix)
  File "/content/eyebrow_poc/data_prep.py", line 210, in process_folder
    for fname in sorted(os.listdir(input_dir)):
                        ^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/차은우'


In [14]:
!python data_prep.py --input_dir "{GOHYUNJUNG_DIR}" --output_dir "{PROCESSED_DIR}" --role celeb --celeb_prefix gohyunjung


W0000 00:00:1787060539.572714    5775 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787060539.588109    5779 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787060539.606905    5780 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Traceback (most recent call last):
  File "/content/eyebrow_poc/data_prep.py", line 295, in <module>
    process_folder(args.input_dir, args.output_dir, args.role, celeb_prefix=args.celeb_prefix)
  File "/content/eyebrow_poc/data_prep.py", line 210, in process_folder
    for fname in sorted(os.listdir(input_dir)):
                        ^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/고윤정'


In [15]:
# 전처리 결과 개수 확인
import json
for role in ["user", "celeb"]:
    path = os.path.join(PROCESSED_DIR, f"manifest_{role}.json")
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(role, "->", len(data), "개")
    else:
        print(role, "-> manifest 없음")


user -> manifest 없음
celeb -> manifest 없음


In [16]:
for p in [GENERAL_FACE_DIR, CHAEUNWOO_DIR, GOHYUNJUNG_DIR]:
    exists = os.path.isdir(p)
    n_files = len(os.listdir(p)) if exists else 0
    print(f"{p} -> 존재={exists}, 파일수={n_files}")

/content/drive/MyDrive/GeneralFace -> 존재=False, 파일수=0
/content/drive/MyDrive/차은우 -> 존재=False, 파일수=0
/content/drive/MyDrive/고윤정 -> 존재=False, 파일수=0


In [17]:
!python data_prep.py --input_dir "{GENERAL_FACE_DIR}" --output_dir "{PROCESSED_DIR}" --role user

W0000 00:00:1787060545.533324    5872 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1787060545.550163    5876 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787060545.569130    5876 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Traceback (most recent call last):
  File "/content/eyebrow_poc/data_prep.py", line 295, in <module>
    process_folder(args.input_dir, args.output_dir, args.role, celeb_prefix=args.celeb_prefix)
  File "/content/eyebrow_poc/data_prep.py", line 210, in process_folder
    for fname in sorted(os.listdir(input_dir)):
                        ^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/GeneralF

In [18]:
#눈썹제거확인
CKPT_EXP1_DIR = "/content/drive/MyDrive/eyebrow_poc_ckpt_exp1"

!python train_exp1_removal.py \
    --data_root "{PROCESSED_DIR}" \
    --epochs 50 \
    --batch_size 8 \
    --out_dir "{CKPT_EXP1_DIR}" \
    --device cuda


Traceback (most recent call last):
  File "/content/eyebrow_poc/train_exp1_removal.py", line 110, in <module>
    main()
  File "/content/eyebrow_poc/train_exp1_removal.py", line 57, in main
    ds = EyebrowUserDataset(args.data_root)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/eyebrow_poc/dataset.py", line 44, in __init__
    with open(manifest_path) as f:
         ^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/eyebrow_poc_processed/manifest_user.json'


In [19]:
#Style Manipulation 확인
CKPT_EXP2_DIR = "/content/drive/MyDrive/eyebrow_poc_ckpt_exp2"
EXP1_CKPT = f"{CKPT_EXP1_DIR}/model_epoch50.pt"

!python train_exp2_style.py \
    --data_root "{PROCESSED_DIR}" \
    --init_ckpt "{EXP1_CKPT}" \
    --epochs 30 \
    --batch_size 8 \
    --out_dir "{CKPT_EXP2_DIR}" \
    --device cuda


Traceback (most recent call last):
  File "/content/eyebrow_poc/train_exp2_style.py", line 154, in <module>
    main()
  File "/content/eyebrow_poc/train_exp2_style.py", line 99, in main
    user_ds = EyebrowUserDataset(args.data_root)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/eyebrow_poc/dataset.py", line 44, in __init__
    with open(manifest_path) as f:
         ^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/eyebrow_poc_processed/manifest_user.json'


In [26]:
#결과확인
import json
with open(os.path.join(PROCESSED_DIR, "manifest_user.json")) as f:
    user_manifest = json.load(f)
print("사용 가능한 stem 목록 (앞 10개):", [e["stem"] for e in user_manifest[:10]])


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
STEM = user_manifest[0]["stem"]  # 원하는 이미지로 교체 가능
RESULTS_DIR = "/content/drive/MyDrive/eyebrow_poc_results"

!python run_experiments.py exp1 \
    --ckpt "{CKPT_EXP1_DIR}/model_epoch50.pt" \
    --data_root "{PROCESSED_DIR}" \
    --stem "{STEM}" \
    --out_dir "{RESULTS_DIR}/exp1"


In [ ]:
!python run_experiments.py exp2 \
    --ckpt "{CKPT_EXP2_DIR}/model_epoch30.pt" \
    --data_root "{PROCESSED_DIR}" \
    --stem "{STEM}" \
    --celeb_ids chaeunwoo gohyunjung \
    --out_dir "{RESULTS_DIR}/exp2"


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def show(path, title):
    img = Image.open(path)
    plt.figure(figsize=(12, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(title)
    plt.show()

show(f"{RESULTS_DIR}/exp1/{STEM}_exp1_compare.png", "Exp1: 원본 | pseudo-removed(참고) | 모델 removal")
show(f"{RESULTS_DIR}/exp2/{STEM}_exp2_styles.png", "Exp2: 원본 | chaeunwoo style | gohyunjung style")


In [ ]:
# 진단: style extractor가 두 사람 눈썹을 애초에 다르게 인코딩하는지 확인
import torch
import torch.nn.functional as F
from dataset import EyebrowStyleRefDataset
from model import EyebrowManipulator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EyebrowManipulator().to(device)
model.load_state_dict(torch.load(f"{CKPT_EXP2_DIR}/model_epoch30.pt", map_location=device))
model.eval()

style_ds = EyebrowStyleRefDataset(PROCESSED_DIR)

chaeunwoo_idxs = style_ds.get_by_celeb_id("chaeunwoo")
gohyunjung_idxs = style_ds.get_by_celeb_id("gohyunjung")
print("chaeunwoo 이미지 수:", len(chaeunwoo_idxs), "| gohyunjung 이미지 수:", len(gohyunjung_idxs))

with torch.no_grad():
    ce_codes = torch.stack([model.style_extractor(style_ds[i]["brow_crop"].unsqueeze(0).to(device))[0]
                             for i in chaeunwoo_idxs[:5]])
    gy_codes = torch.stack([model.style_extractor(style_ds[i]["brow_crop"].unsqueeze(0).to(device))[0]
                             for i in gohyunjung_idxs[:5]])

    within_chaeunwoo = F.cosine_similarity(ce_codes[0:1], ce_codes[1:]).mean().item() if len(ce_codes) > 1 else None
    within_gohyunjung = F.cosine_similarity(gy_codes[0:1], gy_codes[1:]).mean().item() if len(gy_codes) > 1 else None
    between = F.cosine_similarity(ce_codes.mean(0, keepdim=True), gy_codes.mean(0, keepdim=True)).item()

    print(f"chaeunwoo 내부 유사도(같은 사람끼리): {within_chaeunwoo}")
    print(f"gohyunjung 내부 유사도(같은 사람끼리): {within_gohyunjung}")
    print(f"chaeunwoo vs gohyunjung 유사도(다른 사람): {between}")
    print("-> between 값이 1에 가까우면(예: >0.95) style encoder가 둘을 구분 못 하는 것")
    print("-> between 값이 낮은데(예: <0.7)도 최종 이미지가 비슷하면, 문제는 style modulation/decoder 쪽")